In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import torch
import torch.multiprocessing as mp
from jppype import vscode_theme
from torch_geometric.loader import DataLoader
from torch_geometric.transforms import ToDevice

from fundus_toolkits import FundusData
from fundus_vessels_toolkit.models.topology.dataset import BranchDigraphDataset

vscode_theme()
# mp.set_start_method("spawn", force=True)

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

In [ ]:
PATH = [
    Path("/run/media/gaby/GREY SSD/PostDoc/DATA/Fundus/" + folder)
    for folder in ["GAVE-train", "MAPLES-DR", "Fundus-AV", "LES-AV", "INSPIRE", "AV_DRIVE/training"]
]
RAW = [path / "1-images" for path in PATH]
AV = [path / "2-av-pred_CLEMENT" for path in PATH]
TOPO = [path / "3-topo" for path in PATH]

dataset = BranchDigraphDataset.load_from_dirs(RAW, TOPO, AV, resize_to=1024, output_dir="tmp/dataset-test")

In [3]:
dataset = BranchDigraphDataset("ALL_DATA_bundle.tar.gz")

Processing...
Done!


## Graph Augment


In [4]:
ID = 0
m, digraph, _ = dataset.jppype_show(ID, augment=True)
m

warp


[ WARN:0@2.709] global loadsave.cpp:1671 imencodeWithMetadata Unsupported depth image for selected encoder is fallbacked to CV_8U.


GridBox(children=(HTML(value='<h3 style="text-align: center;">001_G/gt</h3>'), HTML(value='<h3 style="text-ali…

warp


BranchDigraphData(edge_index=[2, 19182], pos=[388, 2], edge_dir=[19182, 2], branch_nodes=[388, 2], branch_curves=[388, 20, 2], branch_root_candidates=[388, 2], branch_tip_pos=[388, 2, 2], branch_tip_tan=[388, 2, 2], edge_p=[19182], branch_root_p=[388, 2], branch_fp_p=[388], branch_av_p=[388], branch_dir_p=[388], branch_subtree_idx=[388], img=[3, 1024, 1024], od_yx=[2], mac_yx=[2], vnode_count=405, vnode_coord=[405, 2], name='01_g/fvt', num_nodes=388)

In [ ]:
from fundus_vessels_toolkit.utils.nnet.profiling import Profiler, profiler

Profiler.reset()

dataset.get(20, augment=True, version="fvt")
%timeit dataset.get(20, augment=True, version="fvt")

warp


RuntimeError: The size of tensor a (1024) must match the size of tensor b (2) at non-singleton dimension 1

In [7]:
dataset.get(20, augment=True, version="fvt")
%timeit dataset.get(20, augment=True, version='fvt')

1.18 s ± 18.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
import cProfile

dataset.preload()

In [ ]:
cProfile.run("dataset.get(20, augment=True, version='fvt')", sort="cumulative")

In [ ]:
from fundus_vessels_toolkit.utils.nnet.profiling import Profiler

Profiler.reset()
dataset.get(20, augment=True, version="fvt")
print(Profiler.get("split_branch").print())